# 2.3 Multi-Agent (adaptado a lab local)

**Qué se ve aquí:** patrón *subagents as tools*: un agente principal llama a subagentes envueltos en `@tool`. Cada subagente tiene sus propias tools y su propio contexto.

> En CPU: cada pregunta dispara **≥4 llamadas al modelo** (principal → subagente → tool → subagente → principal). Paciencia.

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [1]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-2/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

## Creating subagents

In [2]:
# Sin cambios
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
#
# # create subagents
#
# subagent_1 = create_agent(
#     model='gpt-5-nano',
#     tools=[square_root]
# )
#
# subagent_2 = create_agent(
#     model='gpt-5-nano',
#     tools=[square]
# )

In [3]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" → MODEL (gemma4 en Ollama, definido en la celda de setup).
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model=MODEL,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=MODEL,
    tools=[square]
)

## Calling subagents

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.messages import HumanMessage
#
# @tool
# def call_subagent_1(x: float) -> float:
#     """Call subagent 1 in order to calculate the square root of a number"""
#     response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
#     return response["messages"][-1].content
#
# @tool
# def call_subagent_2(x: float) -> float:
#     """Call subagent 2 in order to calculate the square of a number"""
#     response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
#     return response["messages"][-1].content
#
# ## Creating the main agent
#
# main_agent = create_agent(
#     model='gpt-5-nano',
#     tools=[call_subagent_1, call_subagent_2],
#     system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

In [4]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" → MODEL (gemma4 en Ollama, definido en la celda de setup).
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=MODEL,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [5]:
# Sin cambios
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [6]:
# Sin cambios
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='ab148bfe-b6ef-4dae-9bc8-5c28f8e01f22'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-05T00:38:05.3239119Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3929751800, 'load_duration': 4139500, 'prompt_eval_count': 155, 'prompt_eval_duration': 3009215000, 'eval_count': 18, 'eval_duration': 901137000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a1097f-0383-7ce1-b61e-f1e8f5dba6e2-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': '6edc1f91-78b9-430e-a2f8-011053f2cbda', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 155, 'output_tokens': 18, 'total_tokens': 173}),
              ToolMessage(content='The square root of 456.0 is approximately 21.354.', name='call_subagent_1', id='5bac9da0-341d